In [1]:

import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import requests

BASE_URL = "https://www.allbirds.com/"
FEED_URL = urljoin(BASE_URL, "products.json")
RAW_DIR = Path("data/raw")

USER_AGENT = (
    "DSC2107-Group9-Course-Project "
    "(contact: REPLACE_WITH_GROUP_EMAIL)"
)

PAGE_SIZE = 250
MAX_REQUESTS = 1000
MAX_RETRIES = 4
TIMEOUT = 25


def check_robots():
    url = urljoin(BASE_URL, "robots.txt")
    response = requests.get(
        url, headers={"User-Agent": USER_AGENT}, timeout=TIMEOUT
    )
    response.raise_for_status()

    parser = RobotFileParser()
    parser.set_url(url)
    parser.parse(response.text.splitlines())

    with open("robots_findings.txt", "w", encoding="utf-8") as f:
        f.write(response.text)

    if not parser.can_fetch(USER_AGENT, FEED_URL):
        raise RuntimeError("robots.txt disallows this feed. Stopping.")

    return True


def request_json(session, params):
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(
                FEED_URL, params=params, timeout=TIMEOUT
            )

            if response.status_code in (401, 403, 429):
                raise RuntimeError(
                    f"Access blocked: HTTP {response.status_code}. Stop."
                )

            response.raise_for_status()
            return response.json()

        except RuntimeError:
            raise
        except (requests.RequestException, ValueError) as error:
            if attempt == MAX_RETRIES - 1:
                raise RuntimeError(
                    f"Request failed after retries: {error}"
                ) from error

            delay = (2 ** attempt) + random.uniform(0.5, 1.5)
            time.sleep(delay)


def main():
    if "cynthiaageno@gmail.com" in USER_AGENT:
        raise ValueError(
            "0791647496"
        )

    check_robots()
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "application/json"
    })

    products = []
    seen_ids = set()
    request_count = 0

    for page in range(1, MAX_REQUESTS + 1):
        if request_count >= MAX_REQUESTS:
            break

        payload = request_json(
            session,
            {"limit": PAGE_SIZE, "page": page}
        )
        request_count += 1

        batch = payload.get("products", [])
        if not isinstance(batch, list) or not batch:
            break

        new_count = 0

        for product in batch:
            product_id = product.get("id")

            if product_id is None:
                # Retain products with missing IDs for investigation.
                products.append(product)
                new_count += 1
            elif product_id not in seen_ids:
                seen_ids.add(product_id)
                products.append(product)
                new_count += 1

        print(
            f"Page {page}: received {len(batch)}, "
            f"new {new_count}, total {len(products)}"
        )

        # Stop if pagination repeats the same products.
        if new_count == 0:
            print("No new products. Pagination may have repeated.")
            break

        if len(batch) < PAGE_SIZE:
            break

        time.sleep(random.uniform(3, 5))

    now_utc = datetime.now(timezone.utc)
    snapshot = {
        "source": FEED_URL,
        "scraped_at": now_utc.isoformat(),
        "request_count": request_count,
        "product_count": len(products),
        "products": products
    }

    day = datetime.now().astimezone().strftime("%Y-%m-%d")
    output = RAW_DIR / f"allbirds_raw_{day}.json"

    # Never overwrite evidence from an earlier run.
    if output.exists():
        output = RAW_DIR / (
            f"allbirds_raw_{day}_"
            f"{datetime.now().strftime('%H%M%S')}.json"
        )

    output.write_text(
        json.dumps(snapshot, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("Saved:", output)
    print("Product count:", len(products))

    if not products:
        print("No data collected. Investigate and document.")
    elif len(products) < 500:
        print(
            "Fewer than 500 products returned. Group 9 must collect "
            "all products actually exposed by its assigned feed."
        )


if __name__ == "__main__":
    main()


Page 1: received 250, new 250, total 250
Page 2: received 250, new 250, total 500
Page 3: received 192, new 192, total 692
Saved: data\raw\allbirds_raw_2026-10-19.json
Product count: 692
